In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

train_df = pd.read_csv('/kaggle/input/mlp-term-3-2025-kaggle-assignment-1/train.csv') 
test_df = pd.read_csv('/kaggle/input/mlp-term-3-2025-kaggle-assignment-1/test.csv')
sample_submission = pd.read_csv('/kaggle/input/mlp-term-3-2025-kaggle-assignment-1/sample_submission.csv')

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

In [ ]:
# Identify data types of different columns in train dataset
print("Data Types in Train Dataset:")
print(train_df.dtypes)

# For test dataset
print("\nData Types in Test Dataset:")
print(test_df.dtypes)

In [ ]:
# Descriptive statistics for numerical columns in train
numerical_cols = train_df.select_dtypes(include=[np.number]).columns
print("Descriptive Statistics for Numerical Columns:")
print(train_df[numerical_cols].describe())

# Note: 'total_sqft', 'bath', 'balcony' are numerical but may have missing/NaN; 'price' is target

In [ ]:
# Descriptive statistics for numerical columns in train
numerical_cols = train_df.select_dtypes(include=[np.number]).columns
print("Descriptive Statistics for Numerical Columns:")
print(train_df[numerical_cols].describe())

# Note: 'total_sqft', 'bath', 'balcony' are numerical but may have missing/NaN; 'price' is target

In [ ]:
# Identify duplicates
print("Duplicates in Train:", train_df.duplicated().sum())
print("Duplicates in Test:", test_df.duplicated().sum())

# Drop duplicates if any (keep first)
train_df = train_df.drop_duplicates()
test_df = test_df.drop_duplicates()

print("\nDuplicates After Dropping (Train):", train_df.duplicated().sum())

In [ ]:
# Identify outliers using IQR method for numerical columns
num_cols = ['total_sqft', 'bath', 'balcony']
def detect_outliers(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)][col]
    return len(outliers), lower, upper

outlier_info = {}
for col in num_cols + ['price']:
    if col in train_df.columns:
        num_outliers, lower, upper = detect_outliers(train_df, col)
        outlier_info[col] = {'outliers': num_outliers, 'lower': lower, 'upper': upper}
        print(f"{col}: {num_outliers} outliers (IQR: [{lower:.2f}, {upper:.2f}])")

print("\nOutlier Summary:")
for col, info in outlier_info.items():
    print(f"{col}: {info['outliers']} outliers")

# Handle outliers: Cap them (winsorize) instead of dropping to retain data
# Explanation: Capping preserves data points while mitigating extreme influence on models
for col in num_cols + ['price']:
    if col in train_df.columns:
        Q1 = train_df[col].quantile(0.25)
        Q3 = train_df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        train_df[col] = np.clip(train_df[col], lower, upper)
        if col != 'price':  # Don't clip test price (no price in test)
            test_df[col] = np.clip(test_df[col], lower, upper)

print("\nOutliers handled by capping.")

In [ ]:
# Visualization 1: Histogram of Price Distribution
plt.figure(figsize=(10, 6))
sns.histplot(train_df['price'], kde=True, bins=30)
plt.title('Distribution of House Prices')
plt.xlabel('Price')
plt.ylabel('Frequency')
plt.show()
# Insight: Prices are right-skewed, with most under 200, indicating potential log transformation for modeling.

# Visualization 2: Boxplot of Total Sqft by Area Type
plt.figure(figsize=(10, 6))
sns.boxplot(x='area_type', y='total_sqft', data=train_df)
plt.title('Total Sqft by Area Type')
plt.xticks(rotation=45)
plt.show()
# Insight: 'type_III' areas tend to have larger sqfts, suggesting premium or spacious properties.

# Visualization 3: Scatter Plot of Price vs Total Sqft
plt.figure(figsize=(10, 6))
sns.scatterplot(x='total_sqft', y='price', data=train_df)
plt.title('Price vs Total Sqft')
plt.show()
# Insight: Positive correlation between size and price, but outliers (capped) show non-linear trends at higher values.

# Bonus: Correlation Heatmap
plt.figure(figsize=(8, 6))
corr = train_df[['total_sqft', 'bath', 'balcony', 'price']].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm')
plt.title('Correlation Heatmap of Numerical Features')
plt.show()
# Insight: Strongest correlation is between total_sqft and price (0.6+), guiding feature importance.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer

# Load data (adjust paths if needed for your environment)
train_df = pd.read_csv('/kaggle/input/mlp-term-3-2025-kaggle-assignment-1/train.csv')
test_df = pd.read_csv('/kaggle/input/mlp-term-3-2025-kaggle-assignment-1/test.csv')

# Create combined_df for consistent preprocessing
combined_df = pd.concat([train_df, test_df], axis=0, ignore_index=True)

# Feature Engineering: Handle total_sqft (ranges and units)
def parse_total_sqft(sqft):
    try:
        return float(sqft)
    except ValueError:
        if '-' in str(sqft):
            min_sqft, max_sqft = map(float, str(sqft).split(' - '))
            return (min_sqft + max_sqft) / 2
        else:
            sqft = str(sqft)
            if 'Sq. Meter' in sqft:
                return float(sqft.split('Sq. Meter')[0]) * 10.7639  # Convert to sqft
            elif 'Acres' in sqft:
                return float(sqft.split('Acres')[0]) * 43560
            elif 'Sq. Yards' in sqft:
                return float(sqft.split('Sq. Yards')[0]) * 9
            elif 'Guntha' in sqft:
                return float(sqft.split('Guntha')[0]) * 1089
            elif 'Cents' in sqft:
                return float(sqft.split('Cents')[0]) * 435.56
            elif 'Grounds' in sqft:
                return float(sqft.split('Grounds')[0]) * 2400
            elif 'Perch' in sqft:
                return float(sqft.split('Perch')[0]) * 272.25
            else:
                return np.nan  # Invalid, to be imputed

combined_df['total_sqft'] = combined_df['total_sqft'].apply(parse_total_sqft)

# Impute missing total_sqft with median
imputer_sqft = SimpleImputer(strategy='median')
combined_df['total_sqft'] = imputer_sqft.fit_transform(combined_df[['total_sqft']])

# Extract bedrooms from size
def extract_bhk(size):
    if pd.isna(size):
        return 1  # Default to 1 BHK
    try:
        return int(str(size).split(' ')[0])
    except:
        return 1  # Default for invalid

combined_df['bhk'] = combined_df['size'].apply(extract_bhk)

# Clip BHK outliers
combined_df['bhk'] = np.clip(combined_df['bhk'], 1, 10)

# Availability: Binary (Ready/Immediate vs Future)
combined_df['is_ready'] = combined_df['availability'].apply(lambda x: 1 if 'Ready' in str(x) or 'Immediate' in str(x) else 0)

# Location: Group rare locations and use target encoding
location_counts = combined_df['location'].value_counts()
rare_locations = location_counts[location_counts <= 10].index
combined_df['location'] = combined_df['location'].apply(lambda x: 'other' if x in rare_locations else x)

# Target encoding for location (use train data only for means)
train_temp = combined_df[:len(train_df)]
location_means = train_temp.groupby('location')['price'].mean()
combined_df['location_target'] = combined_df['location'].map(location_means).fillna(location_means.median())

# New features: sqft per bhk, bath ratio
combined_df['sqft_per_bhk'] = combined_df['total_sqft'] / combined_df['bhk']
combined_df['bath_ratio'] = combined_df['bath'] / combined_df['bhk']

# Log-transform numerical features to handle skewness
numerical_features = ['total_sqft', 'sqft_per_bhk', 'bath', 'balcony', 'bhk', 'location_target', 'bath_ratio']
for feature in numerical_features:
    combined_df[feature] = np.log1p(combined_df[feature])

# Split back into train and test
train_processed = combined_df[:len(train_df)].copy()
test_processed = combined_df[len(train_df):].copy()

# Outlier removal based on price_per_sqft (train only)
train_processed['price_per_sqft'] = train_processed['price'] * 100000 / train_processed['total_sqft']  # Price in lakhs
def remove_outliers(df):
    df_out = pd.DataFrame()
    for key, subdf in df.groupby('location'):
        m = np.mean(subdf.price_per_sqft)
        st = np.std(subdf.price_per_sqft)
        reduced_df = subdf[(subdf.price_per_sqft > (m - 2 * st)) & (subdf.price_per_sqft <= (m + 2 * st))]
        df_out = pd.concat([df_out, reduced_df], ignore_index=True)
    return df_out

train_processed = remove_outliers(train_processed)

# Prepare X, y, X_test
feature_columns = ['area_type', 'is_ready', 'location_target', 'total_sqft', 'bath', 'balcony', 'bhk', 'sqft_per_bhk', 'bath_ratio']
X = train_processed[feature_columns]
y = train_processed['price']
X_test = test_processed[feature_columns]

# Diagnostic
print("X shape:", X.shape)
print("X NaN check:\n", X.isnull().sum())
print("X_test shape:", X_test.shape)
print("X_test NaN check:\n", X_test.isnull().sum())
print("Optimized feature engineering completed. Features:", feature_columns)


In [ ]:
'''# Select features
feature_cols = ['area_type', 'is_ready', 'location_freq', 'total_sqft', 'bath', 'balcony', 'bhk']
X = train_df[feature_cols]
y = train_df['price']
X_test = test_df[feature_cols]

# Encode categorical: area_type (low cardinality) - Label Encoding
le = LabelEncoder()
X['area_type'] = le.fit_transform(X['area_type'])
X_test['area_type'] = le.transform(X_test['area_type'])

# Explanation: Label Encoding for ordinal-like (area_type); OneHot could be used but increases dims.
# Numerical scaling: StandardScaler for models sensitive to scale (e.g., SVM, KNN)
scaler = StandardScaler()
num_features = ['total_sqft', 'bath', 'balcony', 'bhk', 'location_freq']
X[num_features] = scaler.fit_transform(X[num_features])
X_test[num_features] = scaler.transform(X_test[num_features])

print("Features encoded and scaled.")'''
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer

# Load data
train_df = pd.read_csv('/kaggle/input/mlp-term-3-2025-kaggle-assignment-1/train.csv')
test_df = pd.read_csv('/kaggle/input/mlp-term-3-2025-kaggle-assignment-1/test.csv')

# Create combined_df for consistent preprocessing
combined_df = pd.concat([train_df, test_df], axis=0, ignore_index=True)

# Feature Engineering: Handle total_sqft
def parse_total_sqft(sqft):
    try:
        return float(sqft)
    except ValueError:
        sqft = str(sqft)
        if '-' in sqft:
            min_sqft, max_sqft = map(float, sqft.split(' - '))
            return (min_sqft + max_sqft) / 2
        else:
            if 'Sq. Meter' in sqft:
                return float(sqft.split('Sq. Meter')[0]) * 10.7639
            elif 'Acres' in sqft:
                return float(sqft.split('Acres')[0]) * 43560
            elif 'Sq. Yards' in sqft:
                return float(sqft.split('Sq. Yards')[0]) * 9
            elif 'Guntha' in sqft:
                return float(sqft.split('Guntha')[0]) * 1089
            elif 'Cents' in sqft:
                return float(sqft.split('Cents')[0]) * 435.56
            elif 'Grounds' in sqft:
                return float(sqft.split('Grounds')[0]) * 2400
            elif 'Perch' in sqft:
                return float(sqft.split('Perch')[0]) * 272.25
            else:
                return np.nan

combined_df['total_sqft'] = combined_df['total_sqft'].apply(parse_total_sqft)
imputer_sqft = SimpleImputer(strategy='median')
combined_df['total_sqft'] = imputer_sqft.fit_transform(combined_df[['total_sqft']])

# Extract bhk from size
def extract_bhk(size):
    if pd.isna(size):
        return 1
    try:
        return int(str(size).split(' ')[0])
    except:
        return 1

combined_df['bhk'] = combined_df['size'].apply(extract_bhk)
combined_df['bhk'] = np.clip(combined_df['bhk'], 1, 10)

# Create is_ready from availability
combined_df['is_ready'] = combined_df['availability'].apply(lambda x: 1 if 'Ready' in str(x) or 'Immediate' in str(x) else 0)

# Location: Target encoding (replace location_freq)
train_temp = combined_df[:len(train_df)]
location_counts = combined_df['location'].value_counts()
rare_locations = location_counts[location_counts <= 10].index
combined_df['location'] = combined_df['location'].apply(lambda x: 'other' if x in rare_locations else x)
location_means = train_temp.groupby('location')['price'].mean()
combined_df['location_target'] = combined_df['location'].map(location_means).fillna(location_means.median())

# New features
combined_df['sqft_per_bhk'] = combined_df['total_sqft'] / combined_df['bhk']
combined_df['bath_ratio'] = combined_df['bath'] / combined_df['bhk']

# Log-transform numerical features
numerical_features = ['total_sqft', 'bath', 'balcony', 'bhk', 'location_target', 'sqft_per_bhk', 'bath_ratio']
for feature in numerical_features:
    combined_df[feature] = np.log1p(combined_df[feature].fillna(combined_df[feature].median()))

# Outlier removal (train only)
train_processed = combined_df[:len(train_df)].copy()
train_processed['price_per_sqft'] = train_processed['price'] * 100000 / np.expm1(train_processed['total_sqft'])
def remove_outliers(df):
    df_out = pd.DataFrame()
    for key, subdf in df.groupby('location'):
        m = np.mean(subdf.price_per_sqft)
        st = np.std(subdf.price_per_sqft)
        reduced_df = subdf[(subdf.price_per_sqft > (m - 2 * st)) & (subdf.price_per_sqft <= (m + 2 * st))]
        df_out = pd.concat([df_out, reduced_df], ignore_index=True)
    return df_out
train_processed = remove_outliers(train_processed)

# Split back
test_processed = combined_df[len(train_df):].copy()

# Prepare X, y, X_test
feature_cols = ['area_type', 'is_ready', 'location_target', 'total_sqft', 'bath', 'balcony', 'bhk', 'sqft_per_bhk', 'bath_ratio']
X = train_processed[feature_cols]
y = train_processed['price']
X_test = test_processed[feature_cols]

# Diagnostic
print("X shape:", X.shape)
print("X NaN check:\n", X.isnull().sum())
print("X_test shape:", X_test.shape)
print("X_test NaN check:\n", X_test.isnull().sum())
print("Features:", feature_cols)


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, r2_score
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd

# Diagnostic: Check for NaN values in X and X_test
print("NaN values in X before processing:")
print(X.isnull().sum())
print("\nNaN values in X_test before processing:")
print(X_test.isnull().sum())

# Update numerical and categorical columns with new features
num_features = ['total_sqft', 'bath', 'balcony', 'bhk', 'location_target', 'sqft_per_bhk', 'bath_ratio']
cat_features = ['area_type', 'is_ready']

# Use RobustScaler for better handling of outliers
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),  # Impute numerical NaNs
            ('scaler', RobustScaler())
        ]), num_features),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),  # Impute categorical NaNs
            ('encoder', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'))  # OneHot
        ]), cat_features)
    ])

# Log-transform target for RMSLE
y_log = np.log1p(y)

# Split data (use y_log)
X_train, X_val, y_train_log, y_val_log = train_test_split(X, y_log, test_size=0.2, random_state=42)

# Dictionary to store models and performances
models = {}
performances = {}

# Add XGBoost to model configs
model_configs = {
    'LinearRegression': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),
    'Lasso': Lasso(alpha=1.0),
    'DecisionTree': DecisionTreeRegressor(random_state=42),
    'RandomForest': RandomForestRegressor(n_estimators=100, random_state=42),
    'GradientBoosting': GradientBoostingRegressor(random_state=42),
    'SVR': SVR(kernel='rbf'),
    'XGBoost': XGBRegressor(random_state=42)
}

# Train each model with a pipeline
for name, model in model_configs.items():
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    
    pipeline.fit(X_train, y_train_log)
    
    y_pred_log = pipeline.predict(X_val)
    
    rmse = np.sqrt(mean_squared_error(y_val_log, y_pred_log))
    r2 = r2_score(y_val_log, y_pred_log)
    
    models[name] = pipeline
    performances[name] = {'RMSE': rmse, 'R2': r2}
    
    print(f"{name} - RMSE (log): {rmse:.4f}, R2 (log): {r2:.4f}")

print("8 Models trained successfully (added XGBoost). Used log1p(price) for training.")

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
import pandas as pd

# Dictionary to store tuned models and performances
tuned_models = {}
tuned_performances = {}

# 1. Ridge Tuning
ridge_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', Ridge())
])
ridge_params = {'model__alpha': [0.01, 0.1, 1.0, 10.0, 100.0]}
ridge_gs = GridSearchCV(ridge_pipeline, ridge_params, cv=5, scoring='neg_mean_squared_error')
ridge_gs.fit(X_train, y_train_log)
best_ridge = ridge_gs.best_estimator_
y_pred_ridge_log = best_ridge.predict(X_val)
rmse_ridge = np.sqrt(mean_squared_error(y_val_log, y_pred_ridge_log))
r2_ridge = r2_score(y_val_log, y_pred_ridge_log)
tuned_models['Ridge_Tuned'] = best_ridge
tuned_performances['Ridge_Tuned'] = {'RMSE': rmse_ridge, 'R2': r2_ridge}
print(f"Best Ridge params: {ridge_gs.best_params_}")
print(f"Ridge_Tuned - RMSE (log): {rmse_ridge:.4f}, R2 (log): {r2_ridge:.4f}")

# 2. Lasso Tuning
lasso_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', Lasso())
])
lasso_params = {'model__alpha': [0.01, 0.1, 1.0, 10.0]}
lasso_gs = GridSearchCV(lasso_pipeline, lasso_params, cv=5, scoring='neg_mean_squared_error')
lasso_gs.fit(X_train, y_train_log)
best_lasso = lasso_gs.best_estimator_
y_pred_lasso_log = best_lasso.predict(X_val)
rmse_lasso = np.sqrt(mean_squared_error(y_val_log, y_pred_lasso_log))
r2_lasso = r2_score(y_val_log, y_pred_lasso_log)
tuned_models['Lasso_Tuned'] = best_lasso
tuned_performances['Lasso_Tuned'] = {'RMSE': rmse_lasso, 'R2': r2_lasso}
print(f"Best Lasso params: {lasso_gs.best_params_}")
print(f"Lasso_Tuned - RMSE (log): {rmse_lasso:.4f}, R2 (log): {r2_lasso:.4f}")

# 3. RandomForest Tuning
rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=42))
])
rf_params = {
    'model__n_estimators': [100, 200, 300],
    'model__max_depth': [10, 20, None],
    'model__min_samples_split': [2, 5]
}
rf_gs = GridSearchCV(rf_pipeline, rf_params, cv=5, scoring='neg_mean_squared_error')
rf_gs.fit(X_train, y_train_log)
best_rf = rf_gs.best_estimator_
y_pred_rf_log = best_rf.predict(X_val)
rmse_rf = np.sqrt(mean_squared_error(y_val_log, y_pred_rf_log))
r2_rf = r2_score(y_val_log, y_pred_rf_log)
tuned_models['RandomForest_Tuned'] = best_rf
tuned_performances['RandomForest_Tuned'] = {'RMSE': rmse_rf, 'R2': r2_rf}
print(f"Best RandomForest params: {rf_gs.best_params_}")
print(f"RandomForest_Tuned - RMSE (log): {rmse_rf:.4f}, R2 (log): {r2_rf:.4f}")

# 4. GradientBoosting Tuning
gb_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', GradientBoostingRegressor(random_state=42))
])
gb_params = {
    'model__n_estimators': [100, 200, 300],
    'model__learning_rate': [0.01, 0.05, 0.1],
    'model__max_depth': [3, 5, 7]
}
gb_gs = GridSearchCV(gb_pipeline, gb_params, cv=5, scoring='neg_mean_squared_error')
gb_gs.fit(X_train, y_train_log)
best_gb = gb_gs.best_estimator_
y_pred_gb_log = best_gb.predict(X_val)
rmse_gb = np.sqrt(mean_squared_error(y_val_log, y_pred_gb_log))
r2_gb = r2_score(y_val_log, y_pred_gb_log)
tuned_models['GradientBoosting_Tuned'] = best_gb
tuned_performances['GradientBoosting_Tuned'] = {'RMSE': rmse_gb, 'R2': r2_gb}
print(f"Best GradientBoosting params: {gb_gs.best_params_}")
print(f"GradientBoosting_Tuned - RMSE (log): {rmse_gb:.4f}, R2 (log): {r2_gb:.4f}")

# 5. XGBoost Tuning
xgb_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', XGBRegressor(random_state=42))
])
xgb_params = {
    'model__n_estimators': [100, 200, 300],
    'model__learning_rate': [0.01, 0.05, 0.1],
    'model__max_depth': [3, 5, 7]
}
xgb_gs = GridSearchCV(xgb_pipeline, xgb_params, cv=5, scoring='neg_mean_squared_error')
xgb_gs.fit(X_train, y_train_log)
best_xgb = xgb_gs.best_estimator_
y_pred_xgb_log = best_xgb.predict(X_val)
rmse_xgb = np.sqrt(mean_squared_error(y_val_log, y_pred_xgb_log))
r2_xgb = r2_score(y_val_log, y_pred_xgb_log)
tuned_models['XGBoost_Tuned'] = best_xgb
tuned_performances['XGBoost_Tuned'] = {'RMSE': rmse_xgb, 'R2': r2_xgb}
print(f"Best XGBoost params: {xgb_gs.best_params_}")
print(f"XGBoost_Tuned - RMSE (log): {rmse_xgb:.4f}, R2 (log): {r2_xgb:.4f}")

print("Hyperparameter tuning completed for 5 models (Ridge, Lasso, RandomForest, GradientBoosting, XGBoost).")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Combine performances from original and tuned models
all_performances = {**performances, **tuned_performances}

# Create comparison table
perf_df = pd.DataFrame(all_performances).T
print("Model Performance Comparison (log scale):")
print(perf_df)

# Plot RMSE comparison
plt.figure(figsize=(14, 8))
perf_df['RMSE'].plot(kind='bar', color='skyblue')
plt.title('RMSE Comparison Across Models (log scale)')
plt.ylabel('RMSE')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Plot R2 comparison
plt.figure(figsize=(14, 8))
perf_df['R2'].plot(kind='bar', color='lightgreen')
plt.title('R2 Score Comparison Across Models (log scale)')
plt.ylabel('R2 Score')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Insight: XGBoost and ensemble models show the lowest RMSE and highest R2 on log scale, indicating better handling of price skewness. Tuning significantly reduces RMSE for tree-based models.

In [ ]:
import os
import pandas as pd
import numpy as np

# Select the best model based on lowest RMSE (log scale)
best_model_name = min(all_performances, key=lambda x: all_performances[x]['RMSE'])
best_model = tuned_models.get(best_model_name, models.get(best_model_name))
print(f"Best model selected based on lowest RMSE: {best_model_name} with RMSE (log): {all_performances[best_model_name]['RMSE']:.4f}")

# Fit on full training data (X, y_log)
y_log = np.log1p(y)  # Ensure full y is log-transformed
best_model.fit(X, y_log)

# Make predictions on test set
test_predictions_log = best_model.predict(X_test)

# Inverse transform for original scale
test_predictions = np.expm1(test_predictions_log)

# Ensure non-negative
test_predictions = np.maximum(test_predictions, 0)

# Prepare submission
submission = pd.DataFrame({
    'id': test_df['id'].astype(int),
    'price': test_predictions
})

# Save to Kaggle output directory
submission_path = '/kaggle/working/submission.csv'
submission.to_csv(submission_path, index=False)
print(f"Submission file created at: {submission_path}")

# Verify
if os.path.exists(submission_path):
    print("Submission file confirmed.")
    print(submission.head())
else:
    print("Error: Submission file not created.")

# Final diagnostic
print("Submission shape:", submission.shape)
print("Sample predictions:", test_predictions[:5])